In [23]:
import psycopg2
from psycopg2 import Error

# === 工具函数 ===

def create_db_connection(host_name, port_name, user_name, user_password, dbname):
    try:
        conn = psycopg2.connect(
            dbname=dbname,
            user=user_name,
            password=user_password,
            host=host_name,
            port=port_name,
            options='-c client_encoding=utf8'
        )
        conn.autocommit = True  # 全局默认自动提交
        print("Database connection successful")
        return conn
    except Error as err:
        print(f"Error: '{err}'")
        return None

def execute_query(connection, query):
    try:
        with connection.cursor() as cursor:
            cursor.execute(query)
        print(f"Executed: {query.strip().splitlines()[0]}...")
    except Error as err:
        print(f"Error executing {query.strip().splitlines()[0]}: '{err}'")

def execute_many_queries(connection, queries):
    for query in queries:
        execute_query(connection, query)

In [24]:
connection = create_db_connection("localhost", "5432", "user5", "exp.55555", "db6")

Database connection successful


In [25]:
execute_query(connection, "CREATE SCHEMA IF NOT EXISTS sche1;")

Executed: CREATE SCHEMA IF NOT EXISTS sche1;...


In [26]:
tablespace_conn = create_db_connection("localhost", "5432", "user5", "exp.55555", "db6")
execute_query(tablespace_conn, "CREATE TABLESPACE example1 RELATIVE LOCATION 'example1';")
tablespace_conn.close()

Database connection successful
Executed: CREATE TABLESPACE example1 RELATIVE LOCATION 'example1';...


In [27]:
students_ddl = """
CREATE TABLE sche1.students (
    id    VARCHAR(10) PRIMARY KEY,
    name  VARCHAR(50),
    age   INT
)
TABLESPACE example1
PARTITION BY RANGE (age) (
    PARTITION P1 VALUES LESS THAN (18),
    PARTITION P2 VALUES LESS THAN (21),
    PARTITION P3 VALUES LESS THAN (26),
    PARTITION P4 VALUES LESS THAN (41)
)
ENABLE ROW MOVEMENT;
"""
execute_query(connection, students_ddl)

Executed: CREATE TABLE sche1.students (...


In [28]:
execute_query(connection,
    "INSERT INTO sche1.students VALUES('1001','aerf',10),('1021','beu',19),('1031','cekf',11);"
)

Executed: INSERT INTO sche1.students VALUES('1001','aerf',10),('1021','beu',19),('1031','cekf',11);...


In [29]:
try:
    with connection.cursor() as cursor:
        cursor.execute("SELECT * FROM sche1.students PARTITION(P1);")
        results = cursor.fetchall()
        print("Partition P1 records:", results)
except Error as err:
    print(f"Error reading partition: {err}")

Partition P1 records: [('1001', 'aerf', 10), ('1031', 'cekf', 11)]


In [30]:
execute_many_queries(connection, [
    "DROP TABLE IF EXISTS sche1.students;",
    "DROP TABLESPACE IF EXISTS example1;",
    "DROP SCHEMA IF EXISTS sche1 CASCADE;"
])

Executed: DROP TABLE IF EXISTS sche1.students;...
Executed: DROP TABLESPACE IF EXISTS example1;...
Executed: DROP SCHEMA IF EXISTS sche1 CASCADE;...


In [31]:
execute_many_queries(connection, [
    "ALTER TABLE kk.xk DROP CONSTRAINT IF EXISTS xk_fkey_1;",
    """
    ALTER TABLE kk.xk ADD CONSTRAINT xk_fkey_1
    FOREIGN KEY (xh) REFERENCES kk.xs(xh)
    ON UPDATE CASCADE ON DELETE CASCADE;
    """,
    "UPDATE kk.xs SET xh = '1001' WHERE xh = '1437120165';",
    "DELETE FROM kk.xs WHERE xh = '1001';"
])

Executed: ALTER TABLE kk.xk DROP CONSTRAINT IF EXISTS xk_fkey_1;...
Executed: ALTER TABLE kk.xk ADD CONSTRAINT xk_fkey_1...
Executed: UPDATE kk.xs SET xh = '1001' WHERE xh = '1437120165';...
Executed: DELETE FROM kk.xs WHERE xh = '1001';...


In [32]:
execute_many_queries(connection, [
    "UPDATE kk.xs SET xb = '男' WHERE xb IS NULL;",
    "DELETE FROM kk.xs WHERE chrq IS NULL;"
])

Executed: UPDATE kk.xs SET xb = '男' WHERE xb IS NULL;...
Executed: DELETE FROM kk.xs WHERE chrq IS NULL;...


In [33]:
execute_many_queries(connection, [
    "DELETE FROM kk.xk WHERE xh IN (SELECT xh FROM kk.xs WHERE ydh = 'zy');",
    "DELETE FROM kk.xk WHERE (kcbh, jsbh) IN (SELECT kcbh, bh FROM kk.sk WHERE bh IN (SELECT jsbh FROM kk.js WHERE ydh = 'zy'));",
    "DELETE FROM kk.sk WHERE bh IN (SELECT jsbh FROM kk.js WHERE ydh = 'zy');",
    "DELETE FROM kk.xs WHERE ydh = 'zy';",
    "DELETE FROM kk.js WHERE ydh = 'zy';",
    "DELETE FROM kk.xyb WHERE ydh = 'zy';"
])

Executed: DELETE FROM kk.xk WHERE xh IN (SELECT xh FROM kk.xs WHERE ydh = 'zy');...
Executed: DELETE FROM kk.xk WHERE (kcbh, jsbh) IN (SELECT kcbh, bh FROM kk.sk WHERE bh IN (SELECT jsbh FROM kk.js WHERE ydh = 'zy'));...
Executed: DELETE FROM kk.sk WHERE bh IN (SELECT jsbh FROM kk.js WHERE ydh = 'zy');...
Executed: DELETE FROM kk.xs WHERE ydh = 'zy';...
Executed: DELETE FROM kk.js WHERE ydh = 'zy';...
Executed: DELETE FROM kk.xyb WHERE ydh = 'zy';...


In [34]:
execute_many_queries(connection, [
    "UPDATE kk.xs SET bj = '08012048' WHERE xh IN (SELECT DISTINCT xh FROM kk.xk WHERE cj < 60);",
    "DELETE FROM kk.xs WHERE xh IN (SELECT DISTINCT xh FROM kk.xk WHERE cj < 60);"
])


Executed: UPDATE kk.xs SET bj = '08012048' WHERE xh IN (SELECT DISTINCT xh FROM kk.xk WHERE cj < 60);...
Executed: DELETE FROM kk.xs WHERE xh IN (SELECT DISTINCT xh FROM kk.xk WHERE cj < 60);...


In [35]:
create_indexes = [
    "CREATE INDEX IF NOT EXISTS stu_index ON kk.xs(xh);",
    "CREATE INDEX IF NOT EXISTS ad_index ON kk.xs(xm, bj);",
    "CREATE INDEX IF NOT EXISTS year_index ON kk.xs((EXTRACT(YEAR FROM chrq)));",
    "CREATE UNIQUE INDEX IF NOT EXISTS ydh_index ON kk.xyb(ydh);",
    "CREATE INDEX IF NOT EXISTS odd_ydh_index ON kk.xyb(ydh) WHERE (ydh ~ '^[0-9]+$' AND (ydh::int % 2) = 1);"
]
execute_many_queries(connection, create_indexes)

Executed: CREATE INDEX IF NOT EXISTS stu_index ON kk.xs(xh);...
Executed: CREATE INDEX IF NOT EXISTS ad_index ON kk.xs(xm, bj);...
Executed: CREATE INDEX IF NOT EXISTS year_index ON kk.xs((EXTRACT(YEAR FROM chrq)));...
Executed: CREATE UNIQUE INDEX IF NOT EXISTS ydh_index ON kk.xyb(ydh);...
Executed: CREATE INDEX IF NOT EXISTS odd_ydh_index ON kk.xyb(ydh) WHERE (ydh ~ '^[0-9]+$' AND (ydh::int % 2) = 1);...


In [36]:
alter_indexes = [
    "CREATE INDEX IF NOT EXISTS date_index ON kk.xs(chrq);",
    "CREATE INDEX IF NOT EXISTS name_sex_index ON kk.xs(xm, xb);",
    "CREATE INDEX IF NOT EXISTS idx_xk_jsbh ON kk.xk(jsbh);"
]
execute_many_queries(connection, alter_indexes)

Executed: CREATE INDEX IF NOT EXISTS date_index ON kk.xs(chrq);...
Executed: CREATE INDEX IF NOT EXISTS name_sex_index ON kk.xs(xm, xb);...
Executed: CREATE INDEX IF NOT EXISTS idx_xk_jsbh ON kk.xk(jsbh);...


In [37]:
game_ddl = """
CREATE TABLE IF NOT EXISTS kk.game (
    game_id VARCHAR(10) NOT NULL,
    game_name VARCHAR(100),
    game_time TIMESTAMP NOT NULL,
    xf DECIMAL(5,1) NOT NULL,
    CONSTRAINT game_pkey PRIMARY KEY(game_id),
    CONSTRAINT game_cre_index UNIQUE(xf)
);
"""
execute_query(connection, game_ddl)

Executed: CREATE TABLE IF NOT EXISTS kk.game (...


In [38]:
execute_many_queries(connection, [
    "SET enable_indexscan = OFF;",
    "SET enable_bitmapscan = OFF;"
])

Executed: SET enable_indexscan = OFF;...
Executed: SET enable_bitmapscan = OFF;...


In [39]:
try:
    with connection.cursor() as cursor:
        cursor.execute("EXPLAIN ANALYZE SELECT * FROM kk.xs WHERE EXTRACT(YEAR FROM chrq) = 1998;")
        no_idx_plan = cursor.fetchall()
        print("No-index plan:\n", no_idx_plan)
except Error as err:
    print(f"Error explain no-index: {err}")

No-index plan:
 [('Seq Scan on xs  (cost=0.00..454.81 rows=86 width=42) (actual time=0.144..4.829 rows=805 loops=1)',), ("  Filter: (date_part('year'::text, chrq) = 1998::double precision)",), ('  Rows Removed by Filter: 16382',), ('Total runtime: 5.012 ms',)]


In [40]:
execute_many_queries(connection, [
    "SET enable_indexscan = ON;",
    "SET enable_bitmapscan = ON;"
])


Executed: SET enable_indexscan = ON;...
Executed: SET enable_bitmapscan = ON;...


In [41]:
try:
    with connection.cursor() as cursor:
        cursor.execute("EXPLAIN ANALYZE SELECT * FROM kk.xs WHERE EXTRACT(YEAR FROM chrq) = 1998;")
        with_idx_plan = cursor.fetchall()
        print("With-index plan:\n", with_idx_plan)
except Error as err:
    print(f"Error explain with-index: {err}")


With-index plan:
 [('Bitmap Heap Scan on xs  (cost=4.92..162.34 rows=86 width=42) (actual time=0.160..0.742 rows=805 loops=1)',), ("  Recheck Cond: (date_part('year'::text, chrq) = 1998::double precision)",), ('  Heap Blocks: exact=194',), ('  ->  Bitmap Index Scan on year_index  (cost=0.00..4.90 rows=86 width=0) (actual time=0.123..0.123 rows=989 loops=1)',), ("        Index Cond: (date_part('year'::text, chrq) = 1998::double precision)",), ('Total runtime: 0.993 ms',)]


In [42]:
# -- 删除索引
drop_indexes = [
    "DROP INDEX IF EXISTS kk.stu_index;",
    "DROP INDEX IF EXISTS kk.ad_index;"
]
execute_many_queries(connection, drop_indexes)


Executed: DROP INDEX IF EXISTS kk.stu_index;...
Executed: DROP INDEX IF EXISTS kk.ad_index;...


In [43]:
# === 结束 ===
connection.close()
print("All done!")

All done!
